In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

In [ ]:
# --- 1. CONFIGURACIÓN DE RUTAS (Compatible Windows/Linux) ---
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "raw" / "waste_classification" / "test"
MODELS_DIR = BASE_DIR / "models"
MODEL_PATH = MODELS_DIR / "modelo_residuos_EfficientNetB0_v1.keras"
FIGURES_DIR = BASE_DIR / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Verificamos que existan
if not DATA_DIR.exists():
    raise FileNotFoundError(f"No se encontraron los datos en: {DATA_DIR}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo en: {MODEL_PATH}. ¡Ejecuta el entrenamiento primero!")

print(f"Rutas verificadas.")
print(f"Cargando modelo desde: {MODEL_PATH}")

In [ ]:
# --- 2. CARGA DEL MODELO ---
model = load_model(MODEL_PATH)
print("Modelo EfficientNetB0 cargado exitosamente.")

In [ ]:
# --- 3. PREPARACIÓN DE DATOS (Solo Validación) ---
# Usamos las mismas dimensiones que en el entrenamiento
IMG_SIZE = (224, 224)
BATCH_SIZE = 16 # Podemos usar un batch más grande para inferencia

# Solo necesitamos re-escalar, no hacemos Data Augmentation para evaluar.
# NO se aplica rescale=1./255 porque EfficientNetB0 tiene preprocesamiento interno.
datagen = ImageDataGenerator()

print("Generando set de validación...")
test_gen = datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # ¡IMPORTANTE! No mezclar para que coincidan las etiquetas con la predicción
)

class_labels = list(test_gen.class_indices.keys())
print(f"Clases detectadas: {len(class_labels)}")

In [ ]:
print("Evaluando el modelo con los datos de prueba...")
loss, accuracy = model.evaluate(test_gen)

print(f"\nResultados Finales:")
print(f"   Pérdida (Loss): {loss:.4f}")
print(f"   Precisión (Accuracy): {accuracy*100:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("Generando predicciones detalladas...")
predictions = model.predict(test_gen)
y_pred = np.argmax(predictions, axis=1) # Tomamos la clase con mayor probabilidad
y_true = test_gen.classes               # Las etiquetas reales

# Obtenemos los nombres de las 5 clases automáticamente
class_names = list(test_gen.class_indices.keys())

# --- MATRIZ DE CONFUSIÓN ---
# Cada fila representa la clase real, cada columna la clase predicha.
# Los valores en la diagonal son aciertos; fuera de ella, errores.
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names)
plt.title('Matriz de Confusión — EfficientNetB0')
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción del Modelo')
plt.xticks(rotation=45)
plt.tight_layout()
save_path = FIGURES_DIR / "matriz_confusion_EfficientNetB0.png"
plt.savefig(save_path, dpi=300)
plt.show()

# --- REPORTE DE CLASIFICACIÓN ---
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
print(cm)

In [ ]:
import numpy as np

# Buscamos dónde la predicción no coincide con la realidad
errors = np.where(y_pred != y_true)[0]

print(f"\nTotal de errores encontrados: {len(errors)} de {len(y_true)} imágenes reales.")
print(f"   Precisión Global (Accuracy): {1 - len(errors)/len(y_true):.2%}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Extraemos las rutas de los archivos reales y los nombres de las clases
filepaths = test_gen.filepaths
class_names = list(test_gen.class_indices.keys())

# Cuántos errores queremos ver en pantalla (máximo 5 para no saturar)
num_errores_a_mostrar = min(5, len(errors))

if num_errores_a_mostrar > 0:
    print(f"\nMostrando {num_errores_a_mostrar} ejemplos de clasificaciones incorrectas:")
    plt.figure(figsize=(15, 5))
    
    for i in range(num_errores_a_mostrar):
        # Índice del error en el dataset
        idx = errors[i]
        
        # Ruta de la imagen real
        img_path = filepaths[idx]
        
        # Etiquetas correspondientes
        etiqueta_real = class_names[y_true[idx]]
        etiqueta_predicha = class_names[y_pred[idx]]
        
        # Cargar y mostrar la imagen
        img = mpimg.imread(img_path)
        plt.subplot(1, num_errores_a_mostrar, i + 1)
        plt.imshow(img)
        
        # El título mostrará en rojo la equivocación
        plt.title(f"Real: {etiqueta_real}\nPredicción: {etiqueta_predicha}", color='red')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("¡Excelente trabajo! El modelo no cometió ningún error en el set de pruebas.")

In [ ]:
# Accuracy por clase — permite identificar qué categorías le cuestan más al modelo.
# Esta información es útil para el reporte comparativo entre modelos.
print("\nAccuracy por clase:")
print("-" * 35)
for i, class_name in enumerate(class_names):
    mask = y_true == i
    class_acc = np.mean(y_pred[mask] == y_true[mask])
    print(f"  {class_name:<12}: {class_acc*100:.2f}%")